# 5 — Baseline Ensemble GRU

A baseline GRU ensemble that isolates one question: does averaging several
independently trained GRUs help at all, without any of notebook 6's machinery?

Members are trained until they collectively cover the whole NonSEP training
set. Every member gets the same 118 SEP samples plus a distinct,
non-overlapping NonSEP slice of 590. No bootstrap resampling, no hyperparameter
variety, no synthetic augmentation, no pruning: members differ only in which
NonSEP slice they see and in weight initialization. Predictions are combined by
simple averaging, then Platt-calibrated on the validation split with
`class_weight='balanced'`.

This is the floor notebook 6 has to beat.

**Input:** `./final_split_data_HybridNorm_BestFeatures_Siamese` (notebook 3's
Siamese/Triplet output, the best cleaning method per notebook 4).

**Output:** `./results/gru_baseline_ensemble.txt` and
`./results/gru_baseline_ensemble_members.txt`.

**Runs:** 2.

**Run after:** notebook 3.


## 1. Load the Cleaned Data


In [6]:
import pickle
import numpy as np
import os

def load_split(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data["X"].astype(np.float32), data["y"]

folder = "./final_split_data_HybridNorm_BestFeatures_Siamese"

X_train, y_train = load_split(f"{folder}/train_set.pkl")
X_val,   y_val   = load_split(f"{folder}/val_set.pkl")
X_test,  y_test  = load_split(f"{folder}/test_set.pkl")

train_counts = np.bincount(y_train.astype(int), minlength=2)
val_counts   = np.bincount(y_val.astype(int),   minlength=2)
test_counts  = np.bincount(y_test.astype(int),  minlength=2)

print(f"Train: {X_train.shape} | Non-SEP={train_counts[0]} | SEP={train_counts[1]}")
print(f"Val  : {X_val.shape} | Non-SEP={val_counts[0]} | SEP={val_counts[1]}")
print(f"Test : {X_test.shape} | Non-SEP={test_counts[0]} | SEP={test_counts[1]}")

Train: (12380, 288, 3) | Non-SEP=12262 | SEP=118
Val  : (1780, 288, 3) | Non-SEP=1763 | SEP=17
Test : (3559, 288, 3) | Non-SEP=3525 | SEP=34


## 2. GRU Helper Functions

In [7]:
import torch
import torch.nn as nn
import time
from sklearn.metrics import confusion_matrix, brier_score_loss
import warnings
warnings.filterwarnings('ignore')

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")


def compute_metrics(y_true, y_pred):
    cm             = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm.ravel()

    # Probability of Detection (PoD), MEMPSEP Section 9.2 item 1.
    # Identical formula to recall -- kept as a separate named field so
    # printed output matches the paper's terminology.
    pod       = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    recall    = pod
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    accuracy  = (TP + TN) / (TP + TN + FP + FN)

    tss  = recall - FP / (FP + TN) if (FP + TN) > 0 else 0.0

    # HSS1 (Barnes & Leka 2008): (TP + TN - N) / P  ==  (TP - FP) / (TP + FN)
    hss1 = (TP - FP) / (TP + FN) if (TP + FN) > 0 else 0.0

    denom = ((TP + FN) * (FN + TN) + (TP + FP) * (FP + TN))
    hss2  = 2 * (TP * TN - FP * FN) / denom if denom > 0 else 0.0

    hits_random = (TP + FP) * (TP + FN) / (TP + TN + FP + FN)
    gss = (TP - hits_random) / (TP + FP + FN - hits_random) if (TP + FP + FN - hits_random) > 0 else 0.0

    far = FP / (TP + FP) if (TP + FP) > 0 else float('nan')
    bias = (TP + FP) / (TP + FN) if (TP + FN) > 0 else 0.0

    return {
        'TP': int(TP), 'TN': int(TN), 'FP': int(FP), 'FN': int(FN),
        'tss': tss, 'far': far, 'bias': bias,
        'hss1': hss1, 'hss2': hss2, 'gss': gss,
        'pod': pod, 'recall': recall, 'precision': precision, 'f1': f1, 'accuracy': accuracy
    }


class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1, dropout=0.3):
        super(GRUModel, self).__init__()
        self.gru      = nn.GRU(input_size, hidden_size, num_layers=num_layers,
                               batch_first=True,
                               dropout=dropout if num_layers > 1 else 0)
        self.bn       = nn.BatchNorm1d(hidden_size)
        self.dropout  = nn.Dropout(dropout)
        self.fc1      = nn.Linear(hidden_size, 32)
        self.relu     = nn.ReLU()
        self.dropout2 = nn.Dropout(0.2)
        self.fc2      = nn.Linear(32, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        out    = out[:, -1, :]
        out    = self.bn(out)
        out    = self.dropout(out)
        out    = self.relu(self.fc1(out))
        out    = self.dropout2(out)
        return self.fc2(out).squeeze()


def train_and_evaluate_gru(params, X_train, y_train, X_val, y_val, X_test, y_test, class_ratio=13):
    X_tr = torch.tensor(X_train, dtype=torch.float32).to(device)
    y_tr = torch.tensor(y_train, dtype=torch.float32).to(device)
    X_va = torch.tensor(X_val,   dtype=torch.float32).to(device)
    y_va = torch.tensor(y_val,   dtype=torch.float32).to(device)
    X_te = torch.tensor(X_test,  dtype=torch.float32).to(device)

    input_size = X_train.shape[2]
    model      = GRUModel(input_size,
                          hidden_size = params["units"],
                          num_layers  = params["layers"],
                          dropout     = params["dropout"]).to(device)

    pos_weight = torch.tensor([float(class_ratio)], dtype=torch.float32).to(device)
    criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer  = torch.optim.Adam(model.parameters(), lr=params["lr"])

    batch_size     = params["batch_size"]
    n_samples      = X_tr.shape[0]
    best_val_loss  = float('inf')
    patience_count = 0
    best_state     = None

    t0 = time.time()
    for epoch in range(50):
        model.train()
        perm = torch.randperm(n_samples)
        for i in range(0, n_samples, batch_size):
            idx    = perm[i:i + batch_size]
            xb, yb = X_tr[idx], y_tr[idx]
            optimizer.zero_grad()
            loss   = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_va), y_va).item()

        if val_loss < best_val_loss:
            best_val_loss  = val_loss
            patience_count = 0
            best_state     = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            patience_count += 1
            if patience_count >= 5:
                break

    train_time = time.time() - t0

    model.load_state_dict(best_state)
    model.eval()

    with torch.no_grad():
        y_val_prob = torch.sigmoid(model(X_va)).cpu().numpy()

    t0 = time.time()
    with torch.no_grad():
        y_prob = torch.sigmoid(model(X_te)).cpu().numpy()
    y_pred     = (y_prob >= 0.5).astype(int)
    infer_time = time.time() - t0

    metrics = compute_metrics(y_test.astype(int), y_pred)
    metrics['brier']      = brier_score_loss(y_test, y_prob)
    metrics['train_time'] = train_time
    metrics['infer_time'] = infer_time

    del model, optimizer, criterion, X_tr, y_tr, X_va, y_va, X_te, pos_weight
    if device.type == "mps":
        torch.mps.empty_cache()

    return metrics, y_val_prob, y_prob

Using device: mps


## 3. Baseline Ensemble GRU

Every member gets the same full set of 118 SEP samples, plus a distinct,
non-overlapping 5x-SEP-sized (590) slice of Non-SEP -- enough members are
trained so the slices collectively cover the entire Non-SEP training set
exactly once. Still no bootstrap, no hyperparameter variety, no ADASYN,
no top-K selection: every member uses identical hyperparameters, and the
only differences between members are which Non-SEP slice they see and
random weight initialization. Combined by simple averaging of
probabilities (mean, not a learned voting layer) -- then Platt-calibrated.

In [8]:
import copy
from sklearn.linear_model import LogisticRegression

best_params = {
    "units": 64,
    "layers": 1,
    "dropout": 0.1,
    "lr": 0.001,
    "batch_size": 64
}

# ---------------------------------------------------------
# The Non-SEP partition is fixed once, not reshuffled per run -- only
# retraining varies between runs (weight init, batch order). This
# isolates training-stochasticity variance cleanly, without conflating
# it with a different data partition each time.
# ---------------------------------------------------------
N_RUNS = 2

sep_idx    = np.where(y_train == 1)[0]
nonsep_idx = np.where(y_train == 0)[0]
n_sep      = len(sep_idx)
n_nonsep   = len(nonsep_idx)
chunk_size = 5 * n_sep   # Non-SEP slice size per member

rng = np.random.default_rng(42)
shuffled_nonsep_idx = rng.permutation(nonsep_idx)
nonsep_chunks = [
    shuffled_nonsep_idx[i:i + chunk_size]
    for i in range(0, n_nonsep, chunk_size)
]
K_MEMBERS = len(nonsep_chunks)

covered = sum(len(c) for c in nonsep_chunks)
assert covered == n_nonsep, "chunk coverage mismatch"
assert len(set(np.concatenate(nonsep_chunks))) == n_nonsep, "chunks overlap or miss samples"

print("═" * 70)
print("  Baseline Ensemble GRU (no bootstrap, no hyperparameter")
print("  variety, no ADASYN, no top-K selection)")
print(f"  SEP samples (in every member)      : {n_sep}")
print(f"  Non-SEP samples (total, to cover)  : {n_nonsep}")
print(f"  Non-SEP slice size (5x SEP)        : {chunk_size}")
print(f"  Members (fixed across all runs)    : {K_MEMBERS}  (last slice size: {len(nonsep_chunks[-1])})")
print(f"  Params  : {best_params}  (identical for every member)")
print(f"  Runs    : {N_RUNS}")
print("═" * 70)
print(f"Coverage check passed: all {n_nonsep} Non-SEP samples covered exactly once "
      f"across {K_MEMBERS} members.")

baseline_raw_run_metrics = []
baseline_cal_run_metrics = []
baseline_member_rows_all_runs = []   # every member, every run (for saving)

t_experiment_start = time.time()

for run in range(1, N_RUNS + 1):

    print(f"\n{'#' * 70}")
    print(f"  RUN {run}/{N_RUNS}  (same Non-SEP partition, fresh retraining)")
    print(f"{'#' * 70}")

    val_prob_cols  = []
    test_prob_cols = []
    member_metrics_cols = []
    member_times   = []

    t_ensemble_start = time.time()

    for m_idx, nonsep_chunk in enumerate(nonsep_chunks, start=1):

        t0 = time.time()

        X_train_m = np.concatenate([X_train[sep_idx], X_train[nonsep_chunk]], axis=0)
        y_train_m = np.concatenate([y_train[sep_idx], y_train[nonsep_chunk]], axis=0)

        train_counts_m = np.bincount(y_train_m.astype(int), minlength=2)
        class_ratio_m  = train_counts_m[0] / train_counts_m[1]

        metrics_m, y_val_prob_m, y_test_prob_m = train_and_evaluate_gru(
            copy.deepcopy(best_params),
            X_train_m, y_train_m,
            X_val,     y_val,
            X_test,    y_test,
            class_ratio=class_ratio_m
        )

        val_prob_cols.append(y_val_prob_m)
        test_prob_cols.append(y_test_prob_m)
        member_metrics_cols.append(metrics_m)
        baseline_member_rows_all_runs.append({**metrics_m, "run": run, "member": m_idx})

        member_time = time.time() - t0
        member_times.append(member_time)
        remaining = K_MEMBERS - m_idx
        eta = np.mean(member_times) * remaining

        print(f"  Member {m_idx:>2}/{K_MEMBERS}  |  train={n_sep}+{len(nonsep_chunk)} (ratio {class_ratio_m:.2f}:1)  "
              f"|  own test TSS={metrics_m['tss']:.4f}  |  {member_time:>6.1f}s  "
              f"|  ETA ~{eta:.0f}s ({remaining} left)")

    print(f"\n  Trained {K_MEMBERS} members in {time.time() - t_ensemble_start:.1f}s")

    # -----------------------------------------------------
    # Combine: simple mean of member probabilities -- no learned voting.
    # -----------------------------------------------------
    y_val_prob_ens  = np.mean(np.stack(val_prob_cols,  axis=1), axis=1)
    y_test_prob_ens = np.mean(np.stack(test_prob_cols, axis=1), axis=1)

    y_test_pred_ens_raw = (y_test_prob_ens >= 0.5).astype(int)
    ens_metrics_raw = compute_metrics(y_test.astype(int), y_test_pred_ens_raw)
    ens_metrics_raw['brier'] = brier_score_loss(y_test, y_test_prob_ens)

    calibrator = LogisticRegression(class_weight='balanced')
    calibrator.fit(y_val_prob_ens.reshape(-1, 1), y_val.astype(int))
    y_test_prob_ens_cal = calibrator.predict_proba(y_test_prob_ens.reshape(-1, 1))[:, 1]
    y_test_pred_ens_cal = (y_test_prob_ens_cal >= 0.5).astype(int)
    ens_metrics_cal = compute_metrics(y_test.astype(int), y_test_pred_ens_cal)
    ens_metrics_cal['brier'] = brier_score_loss(y_test, y_test_prob_ens_cal)

    baseline_raw_run_metrics.append(ens_metrics_raw)
    baseline_cal_run_metrics.append(ens_metrics_cal)

    print(f"\n  Run {run} ensemble result (simple probability averaging):")
    print(f"    Raw       : TSS={ens_metrics_raw['tss']:.4f}  Brier={ens_metrics_raw['brier']:.4f}")
    print(f"    Calibrated: TSS={ens_metrics_cal['tss']:.4f}  Brier={ens_metrics_cal['brier']:.4f}")

print(f"\n{'═' * 70}")
print(f"  All {N_RUNS} runs done in {time.time() - t_experiment_start:.1f}s total")
print(f"{'═' * 70}")

avg_keys = ['tss', 'far', 'bias', 'hss1', 'hss2', 'gss',
            'pod', 'recall', 'precision', 'f1', 'accuracy', 'brier']

def avg_metrics(metrics_list, keys):
    return {k: np.nanmean([m[k] for m in metrics_list]) for k in keys}

baseline_raw_avg = avg_metrics(baseline_raw_run_metrics, avg_keys)
baseline_cal_avg = avg_metrics(baseline_cal_run_metrics, avg_keys)

print(f"\nBaseline Ensemble GRU -- average of {N_RUNS} runs")
print("Raw (uncalibrated):")
for k in avg_keys:
    print(f"  {k:<10}: {baseline_raw_avg[k]:.4f}")
print("\nCalibrated (Platt scaling, fit on val set):")
for k in avg_keys:
    print(f"  {k:<10}: {baseline_cal_avg[k]:.4f}")
print(f"\nBrier score -- avg raw: {baseline_raw_avg['brier']:.4f}  |  avg calibrated: {baseline_cal_avg['brier']:.4f}")

══════════════════════════════════════════════════════════════════════
  Baseline Ensemble GRU (no bootstrap, no hyperparameter
  variety, no ADASYN, no top-K selection)
  SEP samples (in every member)      : 118
  Non-SEP samples (total, to cover)  : 12262
  Non-SEP slice size (5x SEP)        : 590
  Members (fixed across all runs)    : 21  (last slice size: 462)
  Params  : {'units': 64, 'layers': 1, 'dropout': 0.1, 'lr': 0.001, 'batch_size': 64}  (identical for every member)
  Runs    : 2
══════════════════════════════════════════════════════════════════════
Coverage check passed: all 12262 Non-SEP samples covered exactly once across 21 members.

######################################################################
  RUN 1/2  (same Non-SEP partition, fresh retraining)
######################################################################
  Member  1/21  |  train=118+590 (ratio 5.00:1)  |  own test TSS=0.6684  |    14.8s  |  ETA ~296s (20 left)
  Member  2/21  |  train=118+590 (rati

## 4. Save Results

In [9]:
RESULTS_DIR = "./results"
os.makedirs(RESULTS_DIR, exist_ok=True)


def save_metrics_csv(filepath, rows, extra_cols=()):
    header = (",".join(extra_cols) + ("," if extra_cols else "") +
              "TP,TN,FP,FN,tss,hss1,hss2,gss,pod,recall,precision,"
              "f1,accuracy,brier,far,bias,train_time,infer_time")
    with open(filepath, "w") as f:
        f.write(header + "\n")
        for r in rows:
            prefix = (",".join(str(r[c]) for c in extra_cols) + ",") if extra_cols else ""
            f.write(
                prefix +
                f"{r['TP']},{r['TN']},{r['FP']},{r['FN']},"
                f"{r['tss']:.6f},{r['hss1']:.6f},{r['hss2']:.6f},{r['gss']:.6f},"
                f"{r['pod']:.6f},{r['recall']:.6f},{r['precision']:.6f},{r['f1']:.6f},{r['accuracy']:.6f},"
                f"{r['brier']:.6f},{r['far']:.6f},{r['bias']:.6f},"
                f"{r.get('train_time', float('nan')):.4f},{r.get('infer_time', float('nan')):.4f}\n"
            )
    print(f"Saved: {filepath}")


# Every member, every run
save_metrics_csv(os.path.join(RESULTS_DIR, "gru_baseline_ensemble_members.txt"),
                  baseline_member_rows_all_runs, extra_cols=("run", "member"))

# Final ensemble result per run (raw + calibrated)
final_rows = []
for run in range(1, N_RUNS + 1):
    final_rows.append({**baseline_raw_run_metrics[run - 1], "run": run, "kind": "raw"})
    final_rows.append({**baseline_cal_run_metrics[run - 1], "run": run, "kind": "calibrated"})
save_metrics_csv(os.path.join(RESULTS_DIR, "gru_baseline_ensemble.txt"), final_rows, extra_cols=("run", "kind"))

Saved: ./results/gru_baseline_ensemble_members.txt
Saved: ./results/gru_baseline_ensemble.txt
